# U.S. Treasury Yield Curve Analysis

This notebook analyzes the U.S. Treasury yield curve to detect inversion signals and their relationship with macroeconomic recession risk.

**Key questions:**
- When does the yield curve invert, and how deep is the inversion?
- How reliably does inversion precede NBER recessions?
- How statistically extreme are current spread levels?

**Data source:** Federal Reserve Economic Data (FRED) via 

## 1. Setup & Data Collection

In [ ]:
import pandas as pd
from fredapi import Fred
import matplotlib.pyplot as plt
import time

fred = Fred(api_key='YOUR_FRED_API_KEY')

gs2 = fred.get_series('GS2', observation_start='2000-01-01')
gs10 = fred.get_series('GS10', observation_start='2000-01-01')

print("2Y:", gs2.tail(3))
print("10Y:", gs10.tail(3))

## 2. 2s10s Spread & Inversion Detection

The 2s10s spread (10Y minus 2Y yield) is one of the most widely tracked recession leading indicators.
A negative spread — where short-term rates exceed long-term rates — signals a yield curve **inversion**.
Historically, sustained inversions have preceded every U.S. recession since the 1970s.

In [ ]:
df = pd.DataFrame({'GS2': gs2, 'GS10': gs10})
df = df.dropna()

df['spread'] = df['GS10'] - df['GS2']
df['inverted'] = df['spread'] < 0

recession = fred.get_series('USREC', observation_start='2000-01-01')
df['recession'] = recession
df['recession'] = df['recession'].fillna(0)

print(df[['GS2', 'GS10', 'spread', 'inverted']].tail(5))
print(f"
Current spread: {df['spread'].iloc[-1]:.2f}%p")
print(f"Currently inverted: {df['inverted'].iloc[-1]}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(df.index, df['spread'].min() - 0.2, df['spread'].max() + 0.2,
                where=df['recession'] == 1,
                color='gray', alpha=0.2, label='Recession (NBER)')
ax.plot(df.index, df['spread'], color='steelblue', linewidth=1.2, label='2s10s Spread')
ax.fill_between(df.index, df['spread'], 0,
                where=df['inverted'],
                color='red', alpha=0.3, label='Inversion')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('U.S. Treasury 2s10s Yield Spread with NBER Recessions (2000–Present)', fontsize=14)
ax.set_ylabel('Spread (%p)')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Yield Curve Shape Analysis

Beyond the 2s10s spread, the full yield curve shape reveals the market's expectation of future rates.
Comparing curves across different rate environments shows how dramatically the shape can shift:
- **Normal (upward sloping):** Long-term yields exceed short-term — healthy growth expectations
- **Flat:** Short and long-term yields converge — uncertainty or tightening cycle
- **Inverted (downward sloping):** Short-term yields exceed long-term — recession signal

In [ ]:
maturities = {
    '3M': 'GS3M', '1Y': 'GS1', '2Y': 'GS2',
    '5Y': 'GS5', '7Y': 'GS7', '10Y': 'GS10',
    '20Y': 'GS20', '30Y': 'GS30'
}

series_dict = {}
for label, code in maturities.items():
    series_dict[label] = fred.get_series(code, observation_start='2000-01-01')
    time.sleep(0.3)

curve_df = pd.DataFrame(series_dict)
curve_df = curve_df.dropna()
print(curve_df.tail(3))

In [ ]:
dates = {
    'Jan 2021 (Normal)': '2021-01-01',
    'Oct 2022 (Flat)': '2022-10-01',
    'Jul 2023 (Inverted)': '2023-07-01',
    'May 2026 (Current)': '2026-05-01',
}
maturity_labels = ['3M', '1Y', '2Y', '5Y', '7Y', '10Y', '20Y', '30Y']
colors = ['steelblue', 'orange', 'red', 'green']

fig, ax = plt.subplots(figsize=(10, 6))
for (label, date), color in zip(dates.items(), colors):
    row = curve_df.loc[date]
    ax.plot(maturity_labels, row[maturity_labels], marker='o',
            label=label, color=color, linewidth=2)

ax.set_title('U.S. Treasury Yield Curve — Shape Comparison', fontsize=14)
ax.set_xlabel('Maturity')
ax.set_ylabel('Yield (%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Z-Score Anomaly Detection

A simple inversion flag tells us *whether* the curve is inverted, but not *how extreme* the inversion is.
A rolling 24-month z-score contextualizes the current spread level within its recent rate environment.

- **Z-score > +2:** Spread is abnormally wide
- **Z-score < -2:** Spread is abnormally narrow or deeply inverted

Using a 24-month window avoids distortion from structurally different rate regimes (e.g., comparing 2023 to the 2010 zero-rate environment).

In [ ]:
window = 24
df['spread_mean'] = df['spread'].rolling(window).mean()
df['spread_std'] = df['spread'].rolling(window).std()
df['z_score'] = (df['spread'] - df['spread_mean']) / df['spread_std']

print(df[['spread', 'z_score']].tail(5))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.fill_between(df.index, df['spread'].min() - 0.2, df['spread'].max() + 0.2,
                 where=df['recession'] == 1,
                 color='gray', alpha=0.2, label='Recession (NBER)')
ax1.plot(df.index, df['spread'], color='steelblue', linewidth=1.2, label='2s10s Spread')
ax1.fill_between(df.index, df['spread'], 0,
                 where=df['inverted'], color='red', alpha=0.3, label='Inversion')
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_ylabel('Spread (%p)')
ax1.set_title('U.S. Treasury 2s10s Spread & Z-Score (2000–Present)', fontsize=14)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.plot(df.index, df['z_score'], color='purple', linewidth=1.2, label='Z-Score (24M Rolling)')
ax2.axhline(2, color='red', linewidth=0.8, linestyle='--', label='±2 Threshold')
ax2.axhline(-2, color='red', linewidth=0.8, linestyle='--')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.3)
ax2.fill_between(df.index, df['z_score'], 2,
                 where=df['z_score'] > 2, color='red', alpha=0.2, label='Anomaly')
ax2.fill_between(df.index, df['z_score'], -2,
                 where=df['z_score'] < -2, color='red', alpha=0.2)
ax2.set_ylabel('Z-Score')
ax2.set_xlabel('Date')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Key Findings

- The 2s10s spread inverted in **2000, 2006–07, and 2022–23**, each preceding or coinciding with economic stress
- The **2022–23 inversion was the deepest on record (~-1.0%p)**, yet no NBER recession has been declared — likely due to the Fed's rapid pivot and labor market resilience
- The rolling z-score identifies **2008 and 2020–21** as periods of abnormally wide spreads (post-crisis easing), and **2011 and 2022–23** as abnormally narrow/negative
- Current spread (May 2026): **+0.48%p** — normalized but still below pre-2022 historical averages